# Protein Gym Tutorial with PLM Framework

This notebook demonstrates how to use the PLM framework for protein engineering using the S22A1 dataset from Protein Gym.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr, pearsonr

from plm_framework.controller import Controller
from plm_framework.datamodels import Variant, AssayResult, ProposedVariant
from plm_framework.config import get_config
from plm_framework.utils import setup_logging

# Set up logging
logger = setup_logging()

## 1. Load and Explore the Protein Gym Dataset

First, let's load the S22A1 dataset and explore its structure.

In [ ]:
# Define the path to the dataset
dataset_path = "../test_data/S22A1_HUMAN_Yee_2023_activity.csv"

# Load the dataset
df = pd.read_csv(dataset_path)

# Display basic information
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
# Explore DMS score distribution
plt.figure(figsize=(10, 6))
plt.hist(df['DMS_score'], bins=30, alpha=0.7)
plt.axvline(df['DMS_score'].mean(), color='r', linestyle='--', label=f'Mean: {df["DMS_score"].mean():.2f}')
plt.title('Distribution of DMS Scores')
plt.xlabel('DMS Score')
plt.ylabel('Count')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Define Reference Sequence

Let's define the reference sequence for the S22A1 protein.

In [ ]:
# S22A1 reference sequence
reference_sequence = "MPTVDDILEQVGESGWFQKQAFLILCLLSAAFAPICVGIVFLGFTPDHHCQSPGVAELSQRCGWSPAEELNYTVPGLGPAGEAFLGQCRRYEVDWNQSALSCVDPLASLATNRSHLPLGPCQDGWVYDTPGSSIVTEFNLVCADSWKLDLFQSCLNAGFLFGSLGVGYFADRFGRKLCLLGTVLVNAVSGVLMAFSPNYMSMLLFRLLQGLVSKGNWMAGYTLITEFVGSGSRRTVAIMYQMAFTVGLVALTGLAYALPHWRWLQLAVSLPTFLFLLYYWCVPESPRWLLSQKRNTEAIKIMDHIAQKNGKLPPADLKMLSLEEDVTEKLSPSFADLFRTPRLRKRTFILMYLWFTDSVLYQGLILHMGATSGNLYLDFLYSALVEIPGAFIALITIDRVGRIYPMAMSNLLAGAACLVMIFISPDLHWLNIIIMCVGRMGITIAIQMICLVNAELYPTFVRNLGVMVCSSLCDIGGIITPFIVFRLREVWQALPLILFAVLGLLAAGVTLLLPETKGVALPETMKDAENLGRKAKPKENTIYLKVQTSEPSGT"

print(f"Reference sequence length: {len(reference_sequence)}")

## 3. Prepare Data for PLM Framework

Convert the Protein Gym dataset into the format expected by the PLM framework.

In [ ]:
def load_protein_gym_dataset(file_path, reference_sequence=None):
    """Load a Protein Gym dataset and convert to PLM framework format."""
    df = pd.read_csv(file_path)
    
    print(f"Loaded dataset from {file_path} with {len(df)} rows")
    print(f"Dataset columns: {df.columns.tolist()}")
    
    # Check for required columns
    required_columns = ['mutant', 'mutated_sequence', 'DMS_score']
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' not found in dataset")
    
    # Create variants and assay results
    variants = []
    assay_results = []
    
    for i, row in df.iterrows():
        # Create variant
        variant = Variant(
            id=i + 1,
            name=row['mutant'],
            sequence=row['mutated_sequence']
        )
        variants.append(variant)
        
        # Create assay result
        assay_result = AssayResult(
            variant_id=variant.id,
            score=float(row['DMS_score']),
            uncertainty=0.1,  # Default uncertainty
            round_id=0  # Initial round
        )
        assay_results.append(assay_result)
    
    print(f"Created {len(variants)} variants and {len(assay_results)} assay results")
    
    # Show a few examples
    for i in range(min(5, len(variants))):
        print(f"Example {i+1}: Variant ID={variants[i].id}, Name={variants[i].name}, Score={assay_results[i].score:.4f}")
    
    return reference_sequence, variants, assay_results

# Load the dataset
reference_sequence, variants, assay_results = load_protein_gym_dataset(
    dataset_path, 
    reference_sequence=reference_sequence
)

## 4. Split Data into Train and Test Sets

Split the dataset into training and testing sets for evaluation.

In [ ]:
import random

# Set random seed for reproducibility
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)

# Split data into train and test
test_fraction = 0.2
n_variants = len(variants)
indices = list(range(n_variants))
random.shuffle(indices)

test_size = int(n_variants * test_fraction)
test_indices = set(indices[:test_size])
train_indices = set(indices[test_size:])

train_variants = [variants[i] for i in train_indices]
train_assay_results = [assay_results[i] for i in train_indices]
test_variants = [variants[i] for i in test_indices]
test_assay_results = [assay_results[i] for i in test_indices]

print(f"Split dataset into {len(train_variants)} training and {len(test_variants)} test variants")

## 5. Create Configuration

Set up the configuration for the PLM framework.

In [ ]:
# Create output directory
output_dir = Path("../notebook_results/s22a1")
os.makedirs(output_dir, exist_ok=True)

# Create a configuration file
config_path = output_dir / "config.yaml"

with open(config_path, "w") as f:
    f.write("""
model:
  name: facebook/esm2_t33_650M_UR50D
  embedding_dim: 1280
  reduced_dim: 64
  use_pooling: true
  quantize: false
training:
  batch_size: 8
  learning_rate: 0.001
  weight_decay: 1.0e-05
  epochs: 50
  early_stopping: 10
  device: auto
active_learning:
  acquisition: ucb
  batch_size: 10
  temperature: 1.0
  exploration_weight: 2.0
  diversity_weight: 0.5
data:
  db_path: notebook_results/s22a1/data/variants.db
  embedding_cache: notebook_results/s22a1/data/embeddings.h5
  output_dir: notebook_results/s22a1/results
""")

# Load the configuration
config = get_config(config_path)

# Update paths to be relative to the notebook
config.data.db_path = str(output_dir / "data/variants.db")
config.data.embedding_cache = str(output_dir / "data/embeddings.h5")
config.data.output_dir = str(output_dir / "results")

# Create necessary directories
os.makedirs(os.path.dirname(config.data.db_path), exist_ok=True)
os.makedirs(os.path.dirname(config.data.embedding_cache), exist_ok=True)
os.makedirs(config.data.output_dir, exist_ok=True)

print(f"Configuration created at {config_path}")

## 6. Initialize Controller and Start First Round

Initialize the PLM framework controller and start the first round with an initial batch of variants.

In [ ]:
# Create controller
controller = Controller(config)

# Define initial batch size
initial_batch_size = 10

# Select random initial batch
initial_indices = random.sample(range(len(train_variants)), min(initial_batch_size, len(train_variants)))
initial_variants = [train_variants[i] for i in initial_indices]

# Start round 1
controller.start_round("Round 1", "Initial random batch")

# Add initial variants to database
for variant in initial_variants:
    controller.data_manager.add_variant(variant)

# Get corresponding assay results
selected_variant_ids = {v.id for v in initial_variants}
initial_assay_results = [
    AssayResult(
        variant_id=result.variant_id,
        score=result.score,
        uncertainty=result.uncertainty,
        round_id=1  # Update round ID
    )
    for result in train_assay_results
    if result.variant_id in selected_variant_ids
]

# Add assay results to database
controller.add_assay_results(initial_assay_results)

# Complete round 1
controller.complete_round()

print(f"Round 1 completed with {len(initial_variants)} initial variants")

## 7. Fit Initial Model

Train the initial model using the first batch of variants.

In [ ]:
# Fit initial model
controller.fit_model(round_id=1)

print("Initial model fitted")

# Check model parameters
if hasattr(controller.learner.model, 'coef_'):
    print(f"Model coefficients shape: {controller.learner.model.coef_.shape}")
    print(f"First few coefficients: {controller.learner.model.coef_[:5]}")
elif hasattr(controller.learner.model, 'estimators_'):
    print(f"Number of estimators: {len(controller.learner.model.estimators_)}")
    if hasattr(controller.learner.model.estimators_[0], 'coef_'):
        print(f"First estimator coefficients: {controller.learner.model.estimators_[0].coef_[:5]}")

## 8. Evaluate Initial Model on Test Set

Evaluate the initial model on the test set to establish a baseline.

In [ ]:
# Embed test variants
_, test_embeddings = controller.embedder.embed_variants(test_variants)

# Get test scores
test_scores = np.array([result.score for result in test_assay_results])

# Get predictions
predictions = controller.learner.predict(test_embeddings)

# Handle different return types
if isinstance(predictions, tuple):
    test_predictions = predictions[0]
else:
    test_predictions = predictions

# Ensure test_predictions is a 1D array
if hasattr(test_predictions, 'shape') and len(test_predictions.shape) > 1:
    test_predictions = test_predictions.flatten()

# Calculate metrics
r2 = r2_score(test_scores, test_predictions)
rmse = np.sqrt(mean_squared_error(test_scores, test_predictions))
spearman_corr, _ = spearmanr(test_scores, test_predictions)
pearson_corr, _ = pearsonr(test_scores, test_predictions)

print(f"Initial model performance:")
print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Spearman correlation: {spearman_corr:.4f}")
print(f"Pearson correlation: {pearson_corr:.4f}")

In [ ]:
# Plot predictions vs actual
plt.figure(figsize=(10, 6))
# Plot predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(test_scores, test_predictions, alpha=0.5)
plt.plot([min(test_scores), max(test_scores)], [min(test_scores), max(test_scores)], 'r--')
plt.title(f'Initial Model: Predicted vs Actual (R² = {r2:.4f})')
plt.xlabel('Actual DMS Score')
plt.ylabel('Predicted DMS Score')
plt.grid(True, alpha=0.3)
plt.savefig(output_dir / 'initial_model_performance.png')
plt.show()

## 9. Run Active Learning Loop

Now let's run multiple rounds of active learning to improve our model.

In [ ]:
# Define parameters for active learning
n_rounds = 3
batch_size = 10
strategy = "ucb"
temperature = 1.0

# Track metrics across rounds
round_metrics = {
    'round': [1],  # Start with round 1 (initial model)
    'r2': [r2],
    'rmse': [rmse],
    'spearman': [spearman_corr],
    'pearson': [pearson_corr],
    'n_training_samples': [len(initial_variants)]
}

# Keep track of selected variant IDs
selected_variant_ids = {v.id for v in initial_variants}

# Run active learning rounds
for round_num in range(2, n_rounds + 2):  # Start from round 2
    print(f"\n--- Round {round_num} ---")
    
    # Update available variants (remove already selected ones)
    available_variants = [v for v in train_variants if v.id not in selected_variant_ids]
    print(f"Available variants: {len(available_variants)}")
    
    if not available_variants:
        print("No more variants available. Stopping active learning.")
        break
    
    # Start new round
    controller.start_round(f"Round {round_num}", f"Active learning round {round_num}")
    
    # Propose variants
    proposed_variants = controller.propose_variants(
        candidates=available_variants,
        batch_size=min(batch_size, len(available_variants)),
        strategy=strategy,
        temperature=temperature
    )
    
    print(f"Proposed {len(proposed_variants)} variants")
    
    # Get variant IDs from proposed variants
    round_variant_ids = {v.variant_id for v in proposed_variants}
    selected_variant_ids.update(round_variant_ids)
    
    # Get corresponding assay results
    round_assay_results = [
        AssayResult(
            variant_id=result.variant_id,
            score=result.score,
            uncertainty=result.uncertainty,
            round_id=round_num
        )
        for result in train_assay_results
        if result.variant_id in round_variant_ids
    ]
    
    # Add assay results to database
    controller.add_assay_results(round_assay_results)
    print(f"Added {len(round_assay_results)} assay results")
    
    # Complete round
    controller.complete_round()
    
    # Fit model with all data
    controller.fit_model()
    print("Model updated with new data")
    
    # Evaluate updated model
    predictions = controller.learner.predict(test_embeddings)
    
    # Handle different return types
    if isinstance(predictions, tuple):
        test_predictions = predictions[0]
    else:
        test_predictions = predictions
    
    # Ensure test_predictions is a 1D array
    if hasattr(test_predictions, 'shape') and len(test_predictions.shape) > 1:
        test_predictions = test_predictions.flatten()
    
    # Calculate metrics
    r2 = r2_score(test_scores, test_predictions)
    rmse = np.sqrt(mean_squared_error(test_scores, test_predictions))
    spearman_corr, _ = spearmanr(test_scores, test_predictions)
    pearson_corr, _ = pearsonr(test_scores, test_predictions)
    
    # Update metrics
    round_metrics['round'].append(round_num)
    round_metrics['r2'].append(r2)
    round_metrics['rmse'].append(rmse)
    round_metrics['spearman'].append(spearman_corr)
    round_metrics['pearson'].append(pearson_corr)
    round_metrics['n_training_samples'].append(len(selected_variant_ids))
    
    print(f"Round {round_num} performance:")
    print(f"R²: {r2:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Spearman correlation: {spearman_corr:.4f}")
    print(f"Pearson correlation: {pearson_corr:.4f}")
    print(f"Total training samples: {len(selected_variant_ids)}")

# Save the final model
model_path = output_dir / "final_model.pkl"
controller.save_model(model_path)
print(f"\nFinal model saved to {model_path}")

## 10. Visualize Learning Progress

Let's visualize how our model performance improved over the active learning rounds.

In [ ]:
# Convert metrics to DataFrame
metrics_df = pd.DataFrame(round_metrics)
print(metrics_df)

In [ ]:
# Plot metrics over rounds
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# R² plot
axes[0, 0].plot(metrics_df['round'], metrics_df['r2'], 'o-', linewidth=2)
axes[0, 0].set_title('R² Score vs Round')
axes[0, 0].set_xlabel('Round')
axes[0, 0].set_ylabel('R²')
axes[0, 0].grid(True, alpha=0.3)

# RMSE plot
axes[0, 1].plot(metrics_df['round'], metrics_df['rmse'], 'o-', linewidth=2, color='orange')
axes[0, 1].set_title('RMSE vs Round')
axes[0, 1].set_xlabel('Round')
axes[0, 1].set_ylabel('RMSE')
axes[0, 1].grid(True, alpha=0.3)

# Correlation plot
axes[1, 0].plot(metrics_df['round'], metrics_df['spearman'], 'o-', linewidth=2, color='green', label='Spearman')
axes[1, 0].plot(metrics_df['round'], metrics_df['pearson'], 'o-', linewidth=2, color='purple', label='Pearson')
axes[1, 0].set_title('Correlation vs Round')
axes[1, 0].set_xlabel('Round')
axes[1, 0].set_ylabel('Correlation')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Training samples plot
axes[1, 1].plot(metrics_df['round'], metrics_df['n_training_samples'], 'o-', linewidth=2, color='red')
axes[1, 1].set_title('Training Samples vs Round')
axes[1, 1].set_xlabel('Round')
axes[1, 1].set_ylabel('Number of Training Samples')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'learning_progress.png')
plt.show()

## 11. Final Model Evaluation

Let's evaluate the final model and visualize its predictions.

In [ ]:
# Get final predictions
final_predictions = controller.learner.predict(test_embeddings)

# Handle different return types
if isinstance(final_predictions, tuple):
    test_predictions = final_predictions[0]
    test_uncertainties = final_predictions[1] if len(final_predictions) > 1 else None
else:
    test_predictions = final_predictions
    test_uncertainties = None

# Ensure test_predictions is a 1D array
if hasattr(test_predictions, 'shape') and len(test_predictions.shape) > 1:
    test_predictions = test_predictions.flatten()

# Plot final predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(test_scores, test_predictions, alpha=0.5)
plt.plot([min(test_scores), max(test_scores)], [min(test_scores), max(test_scores)], 'r--')
plt.title(f'Final Model: Predicted vs Actual (R² = {r2:.4f})')
plt.xlabel('Actual DMS Score')
plt.ylabel('Predicted DMS Score')
plt.grid(True, alpha=0.3)
plt.savefig(output_dir / 'final_model_performance.png')
plt.show()

## 12. Generate New Proposals

Now let's use our trained model to propose new variants for the next round of experiments.

In [ ]:
# Generate new candidate variants
from plm_framework.utils import generate_mutations

# Generate new candidates with single mutations
n_candidates = 100
n_mutations = 1

print(f"Generating {n_candidates} new candidate variants with {n_mutations} mutations each...")

# Generate mutations
new_candidates = []
for i in range(n_candidates):
    # Generate random mutations
    mutation_details, variant_seq = generate_mutations(
        reference_sequence, n_mutations=n_mutations
    )
    
    # Create variant name from mutation details
    variant_name = "_".join(mutation_details)
    
    # Create variant
    variant = Variant(
        id=i + 10000,  # Use high IDs to avoid conflicts
        name=variant_name,
        sequence="".join(variant_seq),
    )
    new_candidates.append(variant)

print(f"Generated {len(new_candidates)} new candidate variants")
print("First 5 candidates:")
for i in range(min(5, len(new_candidates))):
    print(f"  {new_candidates[i].name}: {new_candidates[i].sequence[:20]}...")

In [ ]:
# Propose new variants using the trained model
controller.start_round("Proposal Round", "New proposals for next experiments")

# Propose variants
proposed_variants = controller.propose_variants(
    candidates=new_candidates,
    batch_size=20,
    strategy="ucb",
    temperature=1.0
)

controller.complete_round()

print(f"Proposed {len(proposed_variants)} new variants for the next round of experiments")

In [ ]:
# Save proposed variants to CSV
proposals_df = pd.DataFrame([
    {
        "variant_id": pv.variant_id,
        "name": next((v.name for v in new_candidates if v.id == pv.variant_id), ""),
        "sequence": next((v.sequence for v in new_candidates if v.id == pv.variant_id), ""),
        "acquisition_score": pv.acquisition_score,
        "predicted_score": pv.predicted_score,
        "uncertainty": pv.uncertainty
    }
    for pv in proposed_variants
])

# Sort by acquisition score
proposals_df = proposals_df.sort_values("acquisition_score", ascending=False).reset_index(drop=True)

# Save to CSV
proposals_path = output_dir / "next_round_proposals.csv"
proposals_df.to_csv(proposals_path, index=False)

print(f"Saved proposals to {proposals_path}")
proposals_df.head()

## 13. Analyze Proposed Variants

Let's analyze the properties of the proposed variants.

In [ ]:
# Plot distribution of predicted scores and uncertainties
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted scores
axes[0].hist(proposals_df['predicted_score'], bins=10, alpha=0.7)
axes[0].axvline(proposals_df['predicted_score'].mean(), color='r', linestyle='--', 
                label=f'Mean: {proposals_df["predicted_score"].mean():.4f}')
axes[0].set_title('Distribution of Predicted Scores')
axes[0].set_xlabel('Predicted Score')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Uncertainties
axes[1].hist(proposals_df['uncertainty'], bins=10, alpha=0.7, color='orange')
# Uncertainties
axes[1].hist(proposals_df['uncertainty'], bins=10, alpha=0.7, color='orange')
axes[1].axvline(proposals_df['uncertainty'].mean(), color='r', linestyle='--', 
                label=f'Mean: {proposals_df["uncertainty"].mean():.4f}')
axes[1].set_title('Distribution of Uncertainties')
axes[1].set_xlabel('Uncertainty')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'proposal_distributions.png')
plt.show()

In [ ]:
# Scatter plot of predicted score vs uncertainty
plt.figure(figsize=(10, 6))
plt.scatter(proposals_df['predicted_score'], proposals_df['uncertainty'], alpha=0.7)
plt.title('Predicted Score vs Uncertainty for Proposed Variants')
plt.xlabel('Predicted Score')
plt.ylabel('Uncertainty')
plt.grid(True, alpha=0.3)
plt.savefig(output_dir / 'score_vs_uncertainty.png')
plt.show()

## 14. Analyze Mutation Patterns

Let's analyze the mutation patterns in the proposed variants.

In [ ]:
# Extract mutation positions and amino acid changes
def parse_mutation(mutation_name):
    """Parse a mutation name like 'A123B' into original AA, position, and new AA."""
    if not mutation_name or '_' in mutation_name:
        # Handle multiple mutations or empty names
        return None
    
    # Extract original AA, position, and new AA
    import re
    match = re.match(r'([A-Z])([0-9]+)([A-Z])', mutation_name)
    if match:
        orig_aa, pos, new_aa = match.groups()
        return orig_aa, int(pos), new_aa
    return None

# Parse mutations
mutation_data = []
for name in proposals_df['name']:
    # Handle multiple mutations
    if '_' in name:
        mutations = name.split('_')
        for mut in mutations:
            parsed = parse_mutation(mut)
            if parsed:
                orig_aa, pos, new_aa = parsed
                mutation_data.append({
                    'position': pos,
                    'original_aa': orig_aa,
                    'new_aa': new_aa,
                    'variant_name': name,
                    'predicted_score': proposals_df[proposals_df['name'] == name]['predicted_score'].values[0]
                })
    else:
        parsed = parse_mutation(name)
        if parsed:
            orig_aa, pos, new_aa = parsed
            mutation_data.append({
                'position': pos,
                'original_aa': orig_aa,
                'new_aa': new_aa,
                'variant_name': name,
                'predicted_score': proposals_df[proposals_df['name'] == name]['predicted_score'].values[0]
            })

# Convert to DataFrame
mutations_df = pd.DataFrame(mutation_data)
print(f"Extracted {len(mutations_df)} mutations from {len(proposals_df)} proposed variants")
mutations_df.head()

In [ ]:
# Analyze mutation positions
if not mutations_df.empty:
    # Count mutations by position
    position_counts = mutations_df['position'].value_counts().sort_index()
    
    # Plot mutation positions
    plt.figure(figsize=(12, 6))
    plt.bar(position_counts.index, position_counts.values)
    plt.title('Mutation Positions in Proposed Variants')
    plt.xlabel('Sequence Position')
    plt.ylabel('Count')
    plt.grid(True, alpha=0.3)
    plt.savefig(output_dir / 'mutation_positions.png')
    plt.show()
    
    # Analyze amino acid substitutions
    aa_substitutions = mutations_df.groupby(['original_aa', 'new_aa']).size().reset_index(name='count')
    aa_substitutions = aa_substitutions.sort_values('count', ascending=False)
    
    print("Top amino acid substitutions:")
    print(aa_substitutions.head(10))
else:
    print("No mutation data available for analysis.")

## 15. Compare Different Acquisition Strategies

Let's compare different acquisition strategies for proposing variants.

In [ ]:
# Define strategies to compare
strategies = ["ucb", "ei", "ts", "diversity"]
batch_size = 10

# Store proposals from each strategy
strategy_proposals = {}

# Generate proposals using each strategy
for strategy_name in strategies:
    print(f"\nProposing variants using {strategy_name} strategy...")
    
    # Start a temporary round
    controller.start_round(f"{strategy_name.upper()} Round", f"Proposals using {strategy_name} strategy")
    
    # Propose variants
    proposed = controller.propose_variants(
        candidates=new_candidates,
        batch_size=batch_size,
        strategy=strategy_name,
        temperature=1.0
    )
    
    # Complete the round
    controller.complete_round()
    
    # Store proposals
    strategy_proposals[strategy_name] = proposed
    
    print(f"Proposed {len(proposed)} variants using {strategy_name} strategy")

In [ ]:
# Compare strategies
strategy_comparison = {}

for strategy_name, proposals in strategy_proposals.items():
    # Extract variant IDs and scores
    variant_ids = [p.variant_id for p in proposals]
    predicted_scores = [p.predicted_score for p in proposals]
    uncertainties = [p.uncertainty for p in proposals]
    
    # Calculate statistics
    strategy_comparison[strategy_name] = {
        'mean_score': np.mean(predicted_scores),
        'max_score': np.max(predicted_scores),
        'min_score': np.min(predicted_scores),
        'std_score': np.std(predicted_scores),
        'mean_uncertainty': np.mean(uncertainties),
        'max_uncertainty': np.max(uncertainties),
        'min_uncertainty': np.min(uncertainties),
        'std_uncertainty': np.std(uncertainties)
    }

# Convert to DataFrame
comparison_df = pd.DataFrame(strategy_comparison).T
comparison_df

In [ ]:
# Visualize strategy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Predicted scores by strategy
axes[0].bar(comparison_df.index, comparison_df['mean_score'], yerr=comparison_df['std_score'], capsize=5)
axes[0].set_title('Mean Predicted Score by Strategy')
axes[0].set_ylabel('Mean Predicted Score')
axes[0].grid(True, alpha=0.3)

# Uncertainties by strategy
axes[1].bar(comparison_df.index, comparison_df['mean_uncertainty'], yerr=comparison_df['std_uncertainty'], 
           capsize=5, color='orange')
axes[1].set_title('Mean Uncertainty by Strategy')
axes[1].set_ylabel('Mean Uncertainty')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'strategy_comparison.png')
plt.show()

## 16. Analyze Overlap Between Strategies

Let's analyze how much overlap exists between different acquisition strategies.

In [ ]:
# Calculate overlap between strategies
strategy_overlap = {}

for i, strategy1 in enumerate(strategies):
    strategy_overlap[strategy1] = {}
    
    # Get variant IDs for strategy1
    ids1 = set(p.variant_id for p in strategy_proposals[strategy1])
    
    for strategy2 in strategies:
        # Get variant IDs for strategy2
        ids2 = set(p.variant_id for p in strategy_proposals[strategy2])
        
        # Calculate overlap
        overlap = len(ids1.intersection(ids2))
        overlap_pct = overlap / len(ids1) * 100 if ids1 else 0
        
        strategy_overlap[strategy1][strategy2] = overlap_pct

# Convert to DataFrame
overlap_df = pd.DataFrame(strategy_overlap)
overlap_df

In [ ]:
# Visualize overlap as a heatmap
plt.figure(figsize=(8, 6))
plt.imshow(overlap_df.values, cmap='viridis', vmin=0, vmax=100)
plt.colorbar(label='Overlap Percentage')
plt.title('Overlap Between Acquisition Strategies')
plt.xticks(range(len(strategies)), strategies)
plt.yticks(range(len(strategies)), strategies)

# Add text annotations
for i in range(len(strategies)):
    for j in range(len(strategies)):
        plt.text(j, i, f"{overlap_df.iloc[i, j]:.1f}%", 
                 ha="center", va="center", color="white" if overlap_df.iloc[i, j] > 50 else "black")

plt.savefig(output_dir / 'strategy_overlap.png')
plt.show()

## 17. Summary and Next Steps

Let's summarize what we've learned and outline next steps for protein engineering with the PLM framework.

### Summary

In this notebook, we demonstrated how to use the PLM framework for protein engineering using the S22A1 dataset from Protein Gym. We:

1. Loaded and explored the Protein Gym dataset
2. Prepared the data for the PLM framework
3. Split the data into training and test sets
4. Created a configuration for the PLM framework
5. Initialized the controller and started the first round
6. Trained an initial model and evaluated its performance
7. Ran multiple rounds of active learning to improve the model
8. Visualized the learning progress across rounds
9. Generated new candidate variants
10. Proposed variants for the next round of experiments
11. Analyzed the proposed variants and mutation patterns
12. Compared different acquisition strategies

### Key Findings

- The model performance improved over multiple rounds of active learning
- Different acquisition strategies proposed different sets of variants
- The UCB strategy balanced exploration and exploitation
- Certain positions in the protein sequence were more frequently targeted for mutation

### Next Steps

1. **Experimental Validation**: Test the proposed variants in the lab to measure their actual properties
2. **Model Refinement**: Update the model with new experimental data
3. **Exploration Strategies**: Experiment with different acquisition strategies and parameters
4. **Multi-Round Optimization**: Continue the active learning loop for multiple rounds
5. **Advanced Mutations**: Explore double or triple mutations for potentially larger improvements
6. **Constraint Integration**: Add biological constraints to guide the search towards viable variants

The PLM framework provides a powerful approach for protein engineering by combining the strengths of protein language models with active learning strategies.